In [0]:
-- Step 1 - Create gold table for vector searching

-- drop table if exists nleshin_catalog.gold_layer.objects_description purge;

create table nleshin_catalog.gold_layer.objects_description (
    object_id string,
    chunk_id string,
    chunk_text_for_embedding string,
    chunk_text_for_returning string,
    sys_inserted_stamp timestamp DEFAULT current_timestamp(),
    job_run_id string
)
CLUSTER BY (object_id)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%python
#  Step 2 - Get data from silver table, separate it on chunks and prepare the embedding vectors

from pyspark.sql import functions as f

df = spark.sql("""
    WITH prepped AS (
        SELECT 
            file_surrogate_key as object_id,
            ai_prep_search(parsed_content) AS ai_prep_search_result
        FROM nleshin_catalog.silver_layer.objects_description_satellite
        -- where sys_inserted_stamp >= dateadd(MINUTE, -cast(:lookback_minutes as int), :date_interval_end) and sys_inserted_stamp < :date_interval_end
    )
    SELECT 
        object_id,
        exploded_chunks.value:chunk_id as chunk_id,
        exploded_chunks.value:chunk_to_embed as chunk_text_for_embedding,
        exploded_chunks.value:chunk_to_retrieve as chunk_text_for_returning,
        current_timestamp() as sys_inserted_stamp,
        :job_run_id as job_run_id
    FROM prepped,
    LATERAL variant_explode(prepped.ai_prep_search_result:document.contents) AS exploded_chunks
    """,
    args={
        # "date_interval_end": dbutils.widgets.get("date_interval_end"),
        # "lookback_minutes": dbutils.widgets.get("lookback_minutes"),
        "job_run_id": dbutils.widgets.get("job_run_id")
    }
)
df.createOrReplaceTempView("temp_view")

In [0]:
--  Step 3 - Delete old rows if rows with the same object_id already exist

MERGE INTO nleshin_catalog.gold_layer.objects_description AS target
USING temp_view AS source
ON target.object_id = source.object_id
WHEN MATCHED THEN DELETE;

In [0]:
--  Step 4 - Insert new and updated rows

insert into nleshin_catalog.gold_layer.objects_description
select * from temp_view;